# Task 1.1 — Core Contribution / Architecture

**Paper:** Algorithms for Learning Kernels Based on Centered Alignment  
**Authors:** Corinna Cortes, Mehryar Mohri, Afshin Rostamizadeh  
**Venue:** Journal of Machine Learning Research (JMLR), 2012  
**Student:** Pintu Singh | Roll No: 230105

## Step-by-Step Method Description

---

### Step 1: Input — Base Kernels and Training Labels
- **What happens:** The method takes as input a set of $p$ pre-defined base kernel matrices $K_1, K_2, \ldots, K_p$ computed over the training data, along with the label vector $y \in \{-1, +1\}^m$.
- **Paper Reference:** Section 1 (Introduction), Section 3 — problem setup. The goal is to find a convex combination $K_\mu = \sum_{k=1}^{p} \mu_k K_k$ where $\mu_k \geq 0$.
- **Purpose:** Instead of manually picking one kernel, we let the algorithm decide how to combine multiple kernels — reducing the burden of kernel selection from the user.

---

### Step 2: Kernel Centering
- **What happens:** Each base kernel matrix $K_k$ is centered in feature space using the centering operation: $K_c = K - \mathbf{1}K/m - K\mathbf{1}/m + \mathbf{1}K\mathbf{1}/m^2$, where $\mathbf{1}$ is the all-ones matrix.
- **Paper Reference:** Section 2, Definition of centered kernel (Eq. 2). The centered version $K_c(x,x') = \langle \phi_c(x), \phi_c(x') \rangle$ where $\phi_c(x) = \phi(x) - \mathbb{E}[\phi]$.
- **Purpose:** Centering removes the mean from the feature space, which makes the alignment measure more meaningful and stable. Uncentered alignment was shown by the authors to be a flawed measure (Section 2.1).

---

### Step 3: Compute Centered Alignment with Target Kernel
- **What happens:** For each base kernel $K_k$, compute its centered alignment with the target kernel $K_y = yy^\top$ (the ideal kernel derived from labels): $\hat{A}(K_k, K_y) = \frac{\langle K_{k,c}, K_{y,c} \rangle_F}{\sqrt{\langle K_{k,c}, K_{k,c} \rangle_F \langle K_{y,c}, K_{y,c} \rangle_F}}$
- **Paper Reference:** Section 2, Eq. (4) — definition of empirical centered alignment $\hat{A}(K, K')$.
- **Purpose:** This scalar score tells us how similar each base kernel is to the ideal label-based kernel — higher alignment means this kernel captures the label structure better.

---

### Step 4a: ALIGN — Independent Weight Assignment
- **What happens:** The simpler algorithm (ALIGN) sets each weight $\mu_k$ proportional to the alignment of that base kernel with the target: $\mu_k = \hat{A}(K_{k,c}, K_{y,c})$, then normalizes so $\|\mu\|_2 = 1$.
- **Paper Reference:** Section 3.2, Algorithm 1 (ALIGN).
- **Purpose:** A fast, closed-form heuristic that does not require any optimization — weights are assigned greedily based on individual kernel quality.

---

### Step 4b: ALIGNF — Joint Weight Optimization via QP
- **What happens:** The stronger algorithm (ALIGNF) determines all weights $\mu$ jointly by solving a Quadratic Program (QP) that maximizes the centered alignment of the combined kernel $K_\mu = \sum_k \mu_k K_k$ with the target kernel $K_y$: $\max_\mu \hat{A}(K_{\mu,c}, K_{y,c})$ subject to $\|\mu\|_2 \leq 1, \mu \geq 0$.
- **Paper Reference:** Section 3.2, Theorem 1 — the QP formulation. The matrix $M$ with $M_{kl} = \langle K_{k,c}, K_{l,c} \rangle_F$ and vector $a$ with $a_k = \langle K_{k,c}, K_{y,c} \rangle_F$ are computed, and the QP becomes: $\max_\mu \frac{a^\top \mu}{\sqrt{\mu^\top M \mu}}$.
- **Purpose:** By considering interactions between all kernels jointly, ALIGNF finds better weights than the greedy ALIGN — at the cost of solving a small QP (size $p \times p$, not $m \times m$).

---

### Step 5: Train SVM on Combined Kernel
- **What happens:** Using the learned weight vector $\mu$, form the final combined kernel matrix $K_\mu = \sum_k \mu_k K_k$. Pass this as a precomputed kernel to a standard SVM or KRR solver.
- **Paper Reference:** Section 3.1 — the two-stage procedure. Stage 1 learns $\mu$; Stage 2 trains a classifier/regressor using $K_\mu$.
- **Purpose:** This separation of kernel learning and hypothesis learning makes the method modular and computationally efficient.

---

### Step 6: Output — Prediction
- **What happens:** The trained SVM with kernel $K_\mu$ is used to classify new test points. For a test point $x$, the kernel vector $[K_\mu(x, x_1), \ldots, K_\mu(x, x_m)]$ is computed using the same learned weights $\mu$ and base kernels.
- **Paper Reference:** Section 5 — empirical results report misclassification error and RMSE using the learned combined kernel.
- **Purpose:** Final prediction on unseen data, completing the pipeline from multiple base kernels to a single trained classifier.

---

## Final Summary Sentence

This paper solves the problem of automatically selecting the best kernel for SVM from a set of base kernels, by introducing **Centered Alignment** as a theoretically sound similarity measure between kernels and the target labels; the authors claim their ALIGNF algorithm is the first to **consistently and significantly outperform the uniform kernel combination baseline** across both classification and regression tasks.